# TRACE-SCV public-data preflight

## Goal

Verify the official public dataset endpoints and pinned revisions, install the reusable package from a writable path, and run a synthetic software check. This notebook does **not** report scientific results.

## Setup

The Kaggle input mount is read-only. The cell below locates the repository and copies it to `/kaggle/working` before installation. Local execution uses the repository in place.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

DOWNLOAD_DATA = False

def find_repository() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates.extend(path.parent.parent for path in kaggle_input.rglob('experiments/pyproject.toml'))
    for candidate in candidates:
        if (candidate / 'experiments' / 'pyproject.toml').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not find a repository containing experiments/pyproject.toml')

source_root = find_repository()
if str(source_root).startswith('/kaggle/input/'):
    repo_root = Path('/kaggle/working/trace-scv')
    if repo_root.exists():
        shutil.rmtree(repo_root)
    shutil.copytree(source_root, repo_root, ignore=shutil.ignore_patterns('.git', '.venv', 'outputs', 'data'))
else:
    repo_root = source_root

package_root = repo_root / 'experiments'
print('Repository:', repo_root)
print('Package:', package_root)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', str(package_root)], check=True)
print('Installed TRACE-SCV with a regular, non-editable install.')

## Steps

### 1. Check official public releases

In [ ]:
from trace_scv.data.preflight import run_preflight

preflight = run_preflight()
summary = [
    {
        'dataset': row['name'],
        'accessible': row['accessible'],
        'gated': row.get('gated'),
        'revision': row.get('revision'),
        'license': row.get('license_reported') or row.get('license_expected'),
    }
    for row in preflight['datasets']
]
summary

In [ ]:
output_root = package_root / 'outputs' / 'kaggle_preflight'
output_root.mkdir(parents=True, exist_ok=True)
(output_root / 'public_data_preflight.json').write_text(json.dumps(preflight, indent=2), encoding='utf-8')
assert preflight['status'] == 'pass', preflight
print('Required public releases and pinned revisions passed.')

### 2. Optional pinned snapshot download

This is disabled by default because SciVer contains thousands of media files. Turn it on only when the notebook storage and runtime are ready.

In [ ]:
if DOWNLOAD_DATA:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'huggingface-hub>=0.26,<2'], check=True)
    from trace_scv.data.download import download_huggingface_snapshot
    data_root = Path('/kaggle/working/trace-scv-data') if Path('/kaggle/working').exists() else package_root / 'data' / 'public'
    for dataset_key in ('sciver', 'musciclaims'):
        print(dataset_key, download_huggingface_snapshot(dataset_key, data_root))
else:
    print('Dataset download skipped. Set DOWNLOAD_DATA = True when ready.')

## Checks

Run the deterministic synthetic packet, citation metric, and evidence-intervention check.

In [ ]:
from trace_scv.smoke import run_smoke

smoke = run_smoke(output_root / 'smoke')
assert smoke['status'] == 'pass'
smoke

## Next steps

A passing notebook establishes only endpoint access, revision pinning, installation, and software behavior. Next, download the snapshots, run schema/modality/license audits, and freeze the dataset manifest before any model training.